# CHH vs CDGP — Isolation Forest classifier

This notebook reproduces all machine-learning analyses reported in the paper using the NP cohort only. It covers:
1. Feature preprocessing and genetic score computation
2. Isolation Forest and Random Forest training/evaluation (100 or 1000 repeated random splits)
3. SHAP feature-importance analysis
4. Univariate baseline evaluation (inhibin-B; cryptorchidism + testicular volume)

In [ ]:
import shap
import time
import numpy as np
import pandas as pd
import multiprocessing
import matplotlib.pyplot as plt
from scipy.stats import ranksums
import itertools

from joblib import Parallel, delayed
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (confusion_matrix, auc, roc_auc_score,
                             balanced_accuracy_score)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

pd.set_option("display.max_rows", 103, "display.max_columns", None, "display.width", None)

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

## Helper utilities

In [ ]:
def divide(n, d):
    """Safe division: returns NaN when denominator is zero or either value is NaN."""
    if np.isnan(n) or np.isnan(d) or (d == 0):
        return np.nan
    return n / d


def intersection(lst1, lst2):
    """Return elements common to both lists, preserving order of lst1."""
    return [v for v in lst1 if v in lst2]


def mean_of_means(df_arrays, n_features):
    """
    Compute the mean absolute SHAP value per feature across repeated runs.
    df_arrays : pd.Series of arrays, one per repeat (shape: n_samples × n_features)
    Returns a 1-D array of length n_features.
    """
    arrays = np.abs(df_arrays.values)
    means  = np.zeros((len(arrays), n_features))
    for i, arr in enumerate(arrays):
        means[i] = np.mean(arr, axis=0)
    return np.mean(means, axis=0)


def std_of_means(df_arrays, n_features):
    """
    Standard error of the mean absolute SHAP value per feature across repeats.
    Divides by sqrt(500) to match the number of trees used for SHAP estimation.
    """
    arrays = np.abs(df_arrays.values)
    means  = np.zeros((len(arrays), n_features))
    for i, arr in enumerate(arrays):
        means[i] = np.std(arr, axis=0)
    return np.mean(means, axis=0) / np.sqrt(500)


def plot_feat_contributions(X, stats_df, operation, extra_suffix):
    """Save a horizontal bar chart of mean |SHAP| values with SEM error bars."""
    plt.figure(figsize=(15, 10))
    plt.barh(X.columns,
             mean_of_means(stats_df["SHAP_mixed"], len(X.columns)),
             xerr=std_of_means(stats_df["SHAP_mixed"], len(X.columns)))
    plt.xlim(0, 0.5)
    plt.xticks(fontsize=16); plt.yticks(fontsize=16)
    plt.title(f"Mean |SHAP| — {operation} {extra_suffix}", fontsize=18)
    plt.savefig(f"figures/SHAP_contrib/feat_contributions_{operation}_{extra_suffix}",
                bbox_inches="tight")
    plt.close()

## Genetic score computation

Three inheritance-aware scoring functions implement the pathogenicity weighting described in the Methods. Each returns a scalar gene score for a given subject.

In [ ]:
def zygocity_count(row):
    """
    Return the effective allele count for a variant row, accounting for
    zygosity and hemizygosity on chrX in males.

    Returns
    -------
    int : 1 for het or hemizygous males on X; 2 for hom; 0 for females on X
    """
    if row["Zyg"] == "het":
        return 1
    elif (row["Sex"] == "Male") and (row["Chr"] == "chrX"):
        return 1          # hemizygous — counts as one allele
    elif (row["Sex"] == "Female") and (row["Chr"] == "chrX"):
        return 0          # carrier female on X: not counted as affected allele
    else:
        return 2          # homozygous autosomal


def p_union(row):
    """
    Combine per-gene pathogenicity probabilities using the inclusion-exclusion
    union rule: P(A∪B∪...) = 1 − ∏(1 − pᵢ).
    Applied across genes to yield a single per-subject genetics feature.
    """
    probs = list(row.values)
    return 1 - np.prod([1 - p for p in probs])


def compute_gene_score_np_recessive_alt_V5(gene, np_df, AN_MT, AN_MT_males):
    """
    Autosomal-recessive gene score for one subject.

    Strategy: consider all pairs of variant alleles (after expanding hom
    variants to two copies). For each pair take the minimum pathogenicity
    level (the weaker variant limits the compound-het severity). Return the
    maximum over all pairs — i.e. the most damaging biallelic combination.

    Only variants with MT_AC ≥ 2 (at least two mutant allele copies) are
    considered, as required for biallelic hits.
    """
    gene_score = 0
    np_df = np_df[np_df["MT_AC"] >= 2]   # filter: need ≥2 mutant allele copies
    for subject in np_df["Subject ID"].unique():
        sub = np_df[np_df["Subject ID"] == subject]
        # expand hom variants to two rows so itertools.product sees both alleles
        sub = sub.loc[sub.index.repeat(sub["hom_count"])].sort_values("Patho level", ascending=False)
        local_scores = []
        for pair in itertools.product(sub["variant"].tolist(), sub["variant"].tolist()):
            pair_df = pd.concat([sub[sub["variant"] == pair[0]],
                                  sub[sub["variant"] == pair[1]]]).reset_index(drop=True)
            # min() = bottleneck pathogenicity for this biallelic combination
            local_scores.append(np.min([pair_df.loc[0, "Patho level"],
                                        pair_df.loc[1, "Patho level"]]))
        gene_score += np.max(local_scores)
    return gene_score


def compute_gene_score_np_dominant_alt_V5(gene, np_df, AN_MT, AN_MT_males):
    """
    Autosomal-dominant gene score for one subject.

    A single pathogenic allele is sufficient; take the highest pathogenicity
    variant for each subject (after hom expansion) and sum across subjects.
    """
    gene_score = 0
    for subject in np_df["Subject ID"].unique():
        sub = np_df[np_df["Subject ID"] == subject]
        sub = sub.loc[sub.index.repeat(sub["hom_count"])].sort_values("Patho level", ascending=False)
        gene_score += sub.reset_index().loc[0, "Patho level"]
    return gene_score


def compute_gene_score_np_XL_alt_V5(gene, np_df, AN_MT, AN_MT_males):
    """
    X-linked gene score for one subject.

    Only hemizygous males are scored (one pathogenic allele → fully penetrant).
    Females are skipped (carrier status not counted as affected).
    """
    gene_score = 0
    for subject in np_df["Subject ID"].unique():
        if np_df[np_df["Subject ID"] == subject]["Sex"].values[0] == "Male":
            sub = np_df[np_df["Subject ID"] == subject]
            sub = sub.loc[sub.index.repeat(sub["hom_count"])].sort_values("Patho level", ascending=False)
            gene_score += sub.reset_index().loc[0, "Patho level"]
        # females: skip (score += 0)
    return gene_score


def assign_score_func_np(gene, subject, sub_np_df, dominance_df, AN_MT, AN_MT_males):
    """
    Dispatch to the correct inheritance-mode scoring function for a gene.
    Inheritance mode (AR / AD / XL) is looked up from dominance_df.
    """
    mode = dominance_df[
        dominance_df["Genes from Young et al, Endocrine Review, April 2019"] == gene
    ]["CURATED inheritance"].values[0]

    cols_rec = ["Patho level", "variant", "ExonicFunc", "Subject ID", "hom_count", "MT_AC", "Sex"]
    cols_dom = ["Patho level", "variant", "ExonicFunc", "Subject ID", "hom_count", "Sex"]

    if mode == "AR":
        return compute_gene_score_np_recessive_alt_V5(gene, sub_np_df[sub_np_df["Gene"] == gene][cols_rec], AN_MT, AN_MT_males)
    elif mode == "AD":
        return compute_gene_score_np_dominant_alt_V5(gene, sub_np_df[sub_np_df["Gene"] == gene][cols_dom], AN_MT, AN_MT_males)
    elif mode == "XL":
        return compute_gene_score_np_XL_alt_V5(gene, sub_np_df[sub_np_df["Gene"] == gene][cols_dom], AN_MT, AN_MT_males)

## Preprocessing

`preprocess` loads the clinical database, optionally joins variant-level genetic scores, encodes categorical variables, and returns feature matrix `X`, label vector `Y`, and the list of NP cohort subject IDs.

In [ ]:
def preprocess(path, columns_to_drop, z_score, patients=0, genetics=0,
               shuffle=0, no_rank=0, drop_non_sequenced=True, diag=0, patho="ACMG"):
    """
    Load and preprocess the clinical database.

    Parameters
    ----------
    path : str
        Path to the Excel workbook (sheet: 'DP_fusion_format').
    columns_to_drop : list of str
        Feature columns to exclude from X (varies per model variant).
    z_score : int
        Unused legacy parameter (kept for API compatibility).
    patients : int
        0 = all patients; 1 = exclude complete CHH; 2 = exclude partial CHH.
    genetics : bool
        If True, join variant-level data and compute the GENETICS feature.
    shuffle : bool
        If True, randomly permute Gene_score (sanity-check / null model).
    no_rank : bool
        If True, compute per-gene pathogenicity scores (see assign_score_func_np)
        instead of using a single max-variant score as the GENETICS feature.
    drop_non_sequenced : bool
        If True, drop subjects without genetic sequencing data.
    diag : bool
        If True, use diagnostic inheritance column instead of curated one.
    patho : str
        Pathogenicity source: 'ACMG' (InterVar) or 'AlphaM' (AlphaMissense).

    Returns
    -------
    X : pd.DataFrame  — feature matrix (subjects × features)
    Y : pd.Series     — binary label (1 = CHH, 0 = CDGP)
    NP_subjects : list — subject IDs belonging to the NP cohort
    """
    df = pd.read_excel(path, sheet_name="DP_fusion_format")
    df = df.drop(["Asso_Pheno", "Olfaction test"], axis=1)
    df = df[df.filter(regex="^(?!Unnamed)").columns]

    # NP cohort: subject IDs starting with 'L' and exactly 4 characters long
    NP_subjects = [v for v in df["Pt"] if (v[0] == "L") and (len(v) == 4)]

    # Load gene list and curated inheritance modes
    dominance_df = pd.read_excel("../../helper_scripts/data/OMIM_CHH_genes.xlsx")[
        ["Genes from Young et al, Endocrine Review, April 2019", "CURATED inheritance"]
    ]
    dominance_df["Genes from Young et al, Endocrine Review, April 2019"] = (
        dominance_df["Genes from Young et al, Endocrine Review, April 2019"]
        .apply(lambda x: x.split(" ")[0])
    )
    dominance_df = dominance_df.dropna().reset_index(drop=True)
    gene_list = list(dominance_df["Genes from Young et al, Endocrine Review, April 2019"])

    if drop_non_sequenced:
        df.dropna(subset=["GENETICS(Y/N)"], inplace=True)

    if genetics:
        # Load NP cohort variant file (hg38, QS≥30, AF<0.01, indel padding 25bp)
        var_df = pd.read_excel(
            "known_genes_20230828_variants_V8_MT_hg38_0_01_indel_25_QS_30_dups_AlphaM.xlsx"
        )
        var_df = var_df[
            var_df["Gene"].isin(gene_list) &
            var_df["Subject ID"].isin(df["SLIMS Res"].unique())
        ]

        # Replace missing frequency values with 0
        freq_cols = [c for c in var_df.columns if "gnom" in c]
        var_df[freq_cols] = var_df[freq_cols].replace(".", "0").astype(float)

        # Keep only the read with the highest quality score per variant/subject
        var_df["QS_max"] = var_df.groupby(["variant", "Subject ID"]).QS.transform(max)
        var_df = var_df[var_df["QS"] == var_df["QS_max"]].reset_index()

        # Compute allele counts needed for biallelic scoring
        var_df["hom_count"]      = var_df.apply(zygocity_count, axis=1)
        var_df["comp_het_count"] = var_df.groupby(["Gene", "Subject ID"])["Subject ID"].transform("count")
        var_df["MT_AC"]          = var_df["comp_het_count"] + var_df["hom_count"] - 1

        # Assign pathogenicity level
        # Frameshift variants without an InterVar call are treated as Pathogenic
        var_df.loc[var_df["intervar_20180118"] == ".", "intervar_20180118"] = (
            var_df[var_df["intervar_20180118"] == "."]
            .apply(lambda r: "Pathogenic" if r["ExonicFunc"][:5] == "frame" else "Benign", axis=1)
        )
        if patho == "ACMG":
            var_df["Patho level"] = var_df["intervar_20180118"].replace(
                {"Benign": 0, "Likely benign": 0.25, "Uncertain significance": 0.5,
                 "Likely pathogenic": 0.75, "Pathogenic": 1, ".": 0}
            )
            var_df = var_df[var_df["Patho level"] > 0].reset_index(drop=True)
        elif patho == "AlphaM":
            var_df["Patho level"] = var_df["am_pathogenicity"].fillna(1)

        if shuffle:
            # Permute Gene_score as a null-model sanity check
            var_df["Gene_score"] = np.random.permutation(var_df["Gene_score"].values)

        if no_rank:
            # Compute per-gene pathogenicity scores and combine via p_union
            np_scores_df = pd.DataFrame(
                index=list(df["SLIMS Res"].unique()), columns=gene_list
            ).fillna(0.0)
            for subject in list(df["SLIMS Res"].unique()):
                AN_MT = 2; AN_MT_males = 1   # diploid autosomal; males hemizygous on X
                for gene in var_df.Gene.unique():
                    np_scores_df.at[subject, gene] = assign_score_func_np(
                        gene, subject,
                        var_df[(var_df["Subject ID"] == subject) & (var_df["Gene"] == gene)],
                        dominance_df, AN_MT, AN_MT_males
                    )
                # Collapse per-gene scores to one subject-level genetics feature
                var_df.loc[var_df["Subject ID"] == subject, "GENETICS(Y/N)"] = (
                    p_union(np_scores_df.loc[subject])
                )

        # Merge genetics feature back into clinical dataframe
        df = (df.drop(["GENETICS(Y/N)"], axis=1)
                .merge(var_df.groupby("Subject ID").agg({"GENETICS(Y/N)": "max"}).reset_index(),
                       left_on="SLIMS Res", right_on="Subject ID", how="left")
                .drop("Subject ID", axis=1))
        df["GENETICS(Y/N)"] = df["GENETICS(Y/N)"].fillna(0)
    else:
        df = df.drop(["GENETICS(Y/N)"], axis=1)

    # Patient-group filtering
    if patients == 1:
        df = df[(df["Dx"] != "Complete CHH, Kallmann") & (df["Dx"] != "Complete CHH")]
    elif patients == 2:
        df = df[(df["Dx"] != "Partial CHH") & (df["Dx"] != "Partial CHH, Kallmann")]

    # Encode categorical variables
    df = df.replace(
        ["N", "Y", "CDGP", "Complete CHH, Kallmann", "Partial CHH", "Partial CHH, Kallmann",
         "Complete CHH", "Normosmic", "Anosmic", "Hyposmia", "SS", "Subjective"],
        [0,   1,   0,      1,                         1,            1,
         1,             0,          1,        1,         1,    0]
    ).set_index("Pt")

    # Binary testicular volume feature (TV > 2 mL = pubertal onset)
    df["binary_1st_TV"] = (df["1st_TV"] > 2).astype(int)

    X = df.drop(["Dx", "SLIMS Res"] + columns_to_drop, axis=1)
    # Age and anthropometrics at first visit are always dropped (not available at referral)
    if "Age_1st" in X.columns:
        X = X.drop(["Age_1st", "1st_BMI", "1st_Height", "1st_Wt"], axis=1)

    Y = df.Dx.astype("int")
    return X, Y, NP_subjects

## Model training

Each function trains one model on a single random train/test split, computes classification metrics and SHAP values, and returns a result row. These are called in parallel across `numberOfRepeats` splits by `create_stat_df`.

In [ ]:
def split_and_train_if(X_trainval, Y_trainval, stats_df, repeat, X_test=[], Y_test=[]):
    """
    One repeat of Isolation Forest training and evaluation.

    If X_test is not provided, performs an internal 70/30 stratified split.
    Missing values are imputed with column means computed on the training set
    (to avoid data leakage).

    The confusion matrix is inverted relative to the IF convention because
    IF labels outliers as -1 (= CHH) and inliers as +1 (= CDGP), whereas Y
    uses 1 = CHH, 0 = CDGP.
    """
    if len(X_test) == 0:
        X_trainval, X_test, Y_trainval, Y_test = train_test_split(
            X_trainval.values, Y_trainval, test_size=0.3, stratify=Y_trainval
        )
    else:
        X_trainval, X_test = X_trainval.values, X_test.values

    # Mean imputation — fit on train, apply to test (no leakage)
    mean_train = np.nanmean(X_trainval, axis=0)
    X_trainval[np.where(np.isnan(X_trainval))] = np.take(mean_train, np.where(np.isnan(X_trainval))[1])
    X_test[np.where(np.isnan(X_test))]         = np.take(mean_train, np.where(np.isnan(X_test))[1])

    rfc = IsolationForest(n_estimators=100, n_jobs=-1)
    rfc.fit(X_trainval)
    predictions = rfc.predict(X_test)

    # Build PR/ROC curves by ranking samples by anomaly score (ascending = most anomalous first)
    PR_df = pd.DataFrame(columns=["Anomalies", "Precision", "Recall", "TPR", "FPR"])
    PR_df["Anomalies"] = pd.DataFrame(rfc.decision_function(X_test), Y_test.values).sort_values(0).index
    for i in PR_df.index:
        temp_df = PR_df.Anomalies.iloc[:i+1]
        PR_df.loc[i, "Recall"]    = temp_df.sum() / PR_df.Anomalies.sum()
        PR_df.loc[i, "Precision"] = temp_df.sum() / len(temp_df)
        PR_df.loc[i, "TPR"]       = temp_df.sum() / PR_df.Anomalies.sum()
        PR_df.loc[i, "FPR"]       = (len(temp_df) - temp_df.sum()) / (len(PR_df.Anomalies) - PR_df.Anomalies.sum())

    stats_df.loc[repeat, "PR_AUC"]  = auc(np.concatenate(([0.0], PR_df.Recall.values,    [1.0])).astype(float),
                                           np.concatenate(([1.0], PR_df.Precision.values, [0.0])).astype(float))
    stats_df.loc[repeat, "ROC_AUC"] = auc(PR_df.FPR.values, PR_df.TPR.values)

    # Convert Y (0/1) to IF label convention (-1/+1) for confusion matrix
    Y_test_if = (((Y_test * 2) - 1) * -1).astype(int)
    confusion = confusion_matrix(Y_test_if, predictions, labels=[1, -1])
    TP = confusion[1,1]; TN = confusion[0,0]; FP = confusion[0,1]; FN = confusion[1,0]

    stats_df.loc[repeat, "ACC"]         = (divide(TP, TP+FN) + divide(TN, TN+FP)) / 2
    Precision = divide(TP, TP+FP); Recall = divide(TP, TP+FN)
    stats_df.loc[repeat, "Precision"]   = Precision
    stats_df.loc[repeat, "Recall"]      = Recall
    stats_df.loc[repeat, "F1_Score"]    = divide(2 * Recall * Precision, Recall + Precision)
    stats_df.loc[repeat, "Specificity"] = divide(TN, TN+FP)

    explainer = shap.TreeExplainer(rfc)
    stats_df.loc[repeat, "SHAP_CHH_train"]  = explainer.shap_values(X_trainval[Y_trainval.values==1])
    stats_df.loc[repeat, "SHAP_CDGP_train"] = explainer.shap_values(X_trainval[Y_trainval.values==0])
    stats_df.loc[repeat, "SHAP_CHH_test"]   = explainer.shap_values(X_test[Y_test.values==1])
    stats_df.loc[repeat, "SHAP_CDGP_test"]  = explainer.shap_values(X_test[Y_test.values==0])
    stats_df.loc[repeat, "SHAP_mixed"]      = explainer.shap_values(X_test)

    return stats_df.iloc[repeat]


def split_and_train_rf(X_trainval, Y_trainval, stats_df, repeat, X_test=[], Y_test=[]):
    """
    One repeat of Random Forest training and evaluation.

    Structure mirrors split_and_train_if. predict_proba is used instead of
    decision_function to rank samples for the PR/ROC curves.
    SHAP values at index [1] correspond to the positive class (CHH).
    """
    if len(X_test) == 0:
        X_trainval, X_test, Y_trainval, Y_test = train_test_split(
            X_trainval.values, Y_trainval, test_size=0.3, stratify=Y_trainval
        )
    else:
        X_trainval, X_test = X_trainval.values, X_test.values

    mean_train = np.nanmean(X_trainval, axis=0)
    X_trainval[np.where(np.isnan(X_trainval))] = np.take(mean_train, np.where(np.isnan(X_trainval))[1])
    X_test[np.where(np.isnan(X_test))]         = np.take(mean_train, np.where(np.isnan(X_test))[1])

    rfc = RandomForestClassifier(n_estimators=100, n_jobs=-1)
    rfc.fit(X_trainval, Y_trainval)
    predictions = rfc.predict(X_test)

    PR_df = pd.DataFrame(columns=["Anomalies", "Precision", "Recall", "TPR", "FPR"])
    PR_df["Anomalies"] = pd.DataFrame(rfc.predict_proba(X_test), Y_test.values).sort_values(0).index
    for i in PR_df.index:
        temp_df = PR_df.Anomalies.iloc[:i+1]
        PR_df.loc[i, "Recall"]    = temp_df.sum() / PR_df.Anomalies.sum()
        PR_df.loc[i, "Precision"] = temp_df.sum() / len(temp_df)
        PR_df.loc[i, "TPR"]       = temp_df.sum() / PR_df.Anomalies.sum()
        PR_df.loc[i, "FPR"]       = (len(temp_df) - temp_df.sum()) / (len(PR_df.Anomalies) - PR_df.Anomalies.sum())

    stats_df.loc[repeat, "PR_AUC"]  = auc(np.concatenate(([0.0], PR_df.Recall.values,    [1.0])).astype(float),
                                           np.concatenate(([1.0], PR_df.Precision.values, [0.0])).astype(float))
    stats_df.loc[repeat, "ROC_AUC"] = auc(PR_df.FPR.values, PR_df.TPR.values)

    confusion = confusion_matrix(Y_test, predictions, labels=[0, 1])
    TP = confusion[1,1]; TN = confusion[0,0]; FP = confusion[0,1]; FN = confusion[1,0]

    stats_df.loc[repeat, "ACC"]         = (divide(TP, TP+FN) + divide(TN, TN+FP)) / 2
    Precision = divide(TP, TP+FP); Recall = divide(TP, TP+FN)
    stats_df.loc[repeat, "Precision"]   = Precision
    stats_df.loc[repeat, "Recall"]      = Recall
    stats_df.loc[repeat, "F1_Score"]    = divide(2 * Recall * Precision, Recall + Precision)
    stats_df.loc[repeat, "Specificity"] = divide(TN, TN+FP)

    explainer = shap.TreeExplainer(rfc)
    stats_df.loc[repeat, "SHAP_CHH_train"]  = explainer.shap_values(X_trainval[Y_trainval.values==1])
    stats_df.loc[repeat, "SHAP_CDGP_train"] = explainer.shap_values(X_trainval[Y_trainval.values==0])
    stats_df.loc[repeat, "SHAP_CHH_test"]   = explainer.shap_values(X_test[Y_test.values==1])
    stats_df.loc[repeat, "SHAP_CDGP_test"]  = explainer.shap_values(X_test[Y_test.values==0])
    stats_df.loc[repeat, "SHAP_mixed"]      = explainer.shap_values(X_test)[1]   # index 1 = CHH class

    return stats_df.iloc[repeat]


def create_stat_df(X_trainval, Y_trainval, X_test=[], Y_test=[],
                   operation="IF", numberOfRepeats=100, debug=0, rf=0):
    """
    Run `numberOfRepeats` independent train/test splits in parallel and
    aggregate per-repeat metrics into a DataFrame.

    debug=1 disables parallelism (useful for debugging with set_trace).
    rf=1 switches from Isolation Forest to Random Forest.
    """
    print(f"Feature matrix shape: {X_trainval.shape}")
    t0 = time.time()
    stats_df = pd.DataFrame(
        index=np.arange(numberOfRepeats),
        columns=["Precision","Recall","F1_Score","ACC","Specificity",
                 "PR_AUC","ROC_AUC","SHAP_CHH_train","SHAP_CDGP_train",
                 "SHAP_CHH_test","SHAP_CDGP_test","SHAP_mixed"]
    )
    num_cores = multiprocessing.cpu_count()
    train_fn = split_and_train_rf if rf else split_and_train_if

    if debug:
        stats_par = [train_fn(X_trainval, Y_trainval, stats_df, r, X_test, Y_test)
                     for r in range(numberOfRepeats)]
    else:
        stats_par = Parallel(n_jobs=num_cores)(
            delayed(train_fn)(X_trainval, Y_trainval, stats_df, r, X_test, Y_test)
            for r in range(numberOfRepeats)
        )

    t1 = time.time()
    stats_df = pd.DataFrame(stats_par)
    print(f"NaN F1 scores: {stats_df.F1_Score.isna().sum()}  |  elapsed: {t1-t0:.1f}s")
    stats_df.F1_Score.replace(np.nan, 0, inplace=True)
    stats_df = stats_df.infer_objects()
    stats_df["op"] = operation
    return stats_df

## Main analysis — NP cohort (internal cross-validation)

Five model variants are evaluated for each of eight feature sets:

| Variant | Description |
|---------|-------------|
| `IF` | Isolation Forest; GENETICS = max variant pathogenicity score |
| `no_genetics` | IF without any genetics feature |
| `no_rank` | IF with per-gene ACMG pathogenicity scores (p-union) |
| `AlphaM` | IF with per-gene AlphaMissense pathogenicity scores |
| `RF` | Random Forest; same features as `IF` |

In [ ]:
def run_conditions(condition_suffix, diag=0, debug=0, numberOfRepeats=100):
    """
    Run all model variants on the NP cohort with internal cross-validation.

    Parameters
    ----------
    condition_suffix : str   Appended to output filenames for versioning.
    diag : bool              Use diagnostic vs curated inheritance column.
    debug : bool             Disable parallelism (for interactive debugging).
    numberOfRepeats : int    Number of random train/test splits per model.
    """
    data_path = "Data_for_Lausanne_30_06_2023_full.xlsx"

    global_stats_if          = pd.DataFrame()
    global_stats_no_genetics = pd.DataFrame()
    global_stats_no_rank     = pd.DataFrame()
    global_stats_alpha       = pd.DataFrame()
    global_stats_rf          = pd.DataFrame()

    # Each entry defines which columns to drop for a given model variant.
    # The 'subjects' list controls which patient subgroups are included
    # (0=all, 1=exclude complete CHH, 2=exclude partial CHH).
    columns_to_drop_list = [
        [],
        ["FH_DP","FH_CHH","FH_anosmia","Olfaction","Asso_Phe","Crypt","Micro","binary_1st_TV","1st_TV"],
        ["Age_1st","1st_Height","1st_Wt","1st_BMI","1st_T","1st_LH","1st_FSH","1st_INB","1st_AMH","1st_IGF1"],
        ["Olfaction","Age_1st","1st_Height","1st_Wt","1st_BMI","1st_T","1st_LH","1st_FSH","1st_INB","1st_AMH","1st_IGF1"],
        ["Age_1st","1st_Height","1st_Wt","1st_BMI","1st_T","1st_LH","1st_FSH","1st_INB","1st_AMH","1st_IGF1"],
        ["Olfaction","Age_1st","1st_Height","1st_Wt","1st_BMI","1st_T","1st_LH","1st_FSH","1st_INB","1st_AMH","1st_IGF1"],
        ["Age_1st","1st_Height","1st_Wt","1st_BMI","1st_T","1st_LH","1st_FSH","1st_INB","1st_AMH","1st_IGF1"],
        ["Olfaction","Age_1st","1st_Height","1st_Wt","1st_BMI","1st_T","1st_LH","1st_FSH","1st_INB","1st_AMH","1st_IGF1"],
    ]
    operations = ["IF","biochem","Clinical","Clinical - WO Olfaction",
                  "Clinical - Complete CHH","Clinical - Complete CHH - WO Olfaction",
                  "Clinical - Partial CHH","Clinical - Partial CHH - WO Olfaction"]
    subjects = [0, 0, 0, 0, 2, 2, 1, 1]

    for param in zip(columns_to_drop_list, operations, subjects):
        print(param)

        # ── IF: max variant score ────────────────────────────────────────────
        X, Y, NP_subjects = preprocess(data_path, param[0], 1, patients=param[2],
                                        drop_non_sequenced=False, diag=diag, patho="ACMG")
        X_NP = X.loc[intersection(NP_subjects, X.index)]
        Y_NP = Y.loc[intersection(NP_subjects, X.index)]
        stats_df = create_stat_df(X_NP, Y_NP, operation=param[1], numberOfRepeats=numberOfRepeats, debug=debug, rf=0)
        global_stats_if = pd.concat([global_stats_if, stats_df[["op","Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]]])
        print(stats_df[["Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]].agg(["mean","std","sem"]))
        plot_feat_contributions(X_NP, stats_df, param[1], "")

        # ── IF: no genetics feature ──────────────────────────────────────────
        X, Y, NP_subjects = preprocess(data_path, param[0], 1, patients=param[2],
                                        diag=diag, patho="ACMG")
        X_NP = X.loc[intersection(NP_subjects, X.index)]
        Y_NP = Y.loc[intersection(NP_subjects, X.index)]
        stats_df = create_stat_df(X_NP, Y_NP, operation=param[1], numberOfRepeats=numberOfRepeats, debug=debug, rf=0)
        global_stats_no_genetics = pd.concat([global_stats_no_genetics, stats_df[["op","Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]]])
        print(stats_df[["Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]].agg(["mean","std","sem"]))
        plot_feat_contributions(X_NP, stats_df, param[1], "_no_genetics")

        # ── IF: per-gene ACMG pathogenicity scores ───────────────────────────
        X, Y, NP_subjects = preprocess(data_path, param[0], 1, patients=param[2],
                                        genetics=1, no_rank=1, diag=diag, patho="ACMG")
        X_NP = X.loc[intersection(NP_subjects, X.index)]
        Y_NP = Y.loc[intersection(NP_subjects, X.index)]
        stats_df = create_stat_df(X_NP, Y_NP, operation=param[1], numberOfRepeats=numberOfRepeats, debug=debug, rf=0)
        global_stats_no_rank = pd.concat([global_stats_no_rank, stats_df[["op","Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]]])
        print(stats_df[["Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]].agg(["mean","std","sem"]))
        plot_feat_contributions(X_NP, stats_df, param[1], "_no_rank")

        # ── IF: per-gene AlphaMissense pathogenicity scores ──────────────────
        X, Y, NP_subjects = preprocess(data_path, param[0], 1, patients=param[2],
                                        genetics=1, no_rank=1, diag=diag, patho="AlphaM")
        X_NP = X.loc[intersection(NP_subjects, X.index)]
        Y_NP = Y.loc[intersection(NP_subjects, X.index)]
        stats_df = create_stat_df(X_NP, Y_NP, operation=param[1], numberOfRepeats=numberOfRepeats, debug=debug, rf=0)
        global_stats_alpha = pd.concat([global_stats_alpha, stats_df[["op","Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]]])
        print(stats_df[["Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]].agg(["mean","std","sem"]))
        plot_feat_contributions(X_NP, stats_df, param[1], "_alpha")

        # ── Random Forest ────────────────────────────────────────────────────
        X, Y, NP_subjects = preprocess(data_path, param[0], 1, patients=param[2],
                                        drop_non_sequenced=False, diag=diag, patho="ACMG")
        X_NP = X.loc[intersection(NP_subjects, X.index)]
        Y_NP = Y.loc[intersection(NP_subjects, X.index)]
        stats_df = create_stat_df(X_NP, Y_NP, operation=param[1], numberOfRepeats=numberOfRepeats, debug=debug, rf=1)
        global_stats_rf = pd.concat([global_stats_rf, stats_df[["op","Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]]])
        print(stats_df[["Precision","Recall","F1_Score","ACC","Specificity","PR_AUC","ROC_AUC"]].agg(["mean","std","sem"]))
        plot_feat_contributions(X_NP, stats_df, param[1], "_rf")

    # Save results
    suffix = f"{numberOfRepeats}_V8_V5_QS_30_{condition_suffix}"
    global_stats_if.to_excel(         f"results/global_stats_if_{suffix}.xlsx",          index=False)
    global_stats_no_genetics.to_excel(f"results/global_stats_no_genetics_{suffix}.xlsx", index=False)
    global_stats_no_rank.to_excel(    f"results/global_stats_no_rank_{suffix}.xlsx",     index=False)
    global_stats_alpha.to_excel(      f"results/global_stats_alphaM_{suffix}.xlsx",      index=False)
    global_stats_rf.to_excel(         f"results/global_stats_rf_{suffix}.xlsx",          index=False)

### Execute — 100 repeats

In [ ]:
run_conditions(condition_suffix="NP_only", diag=0, numberOfRepeats=100)

In [ ]:
# Extended run with 1000 repeats for final paper figures
run_conditions(condition_suffix="NP_only_1000", diag=0, numberOfRepeats=1000)

## Data inspection

In [ ]:
X, Y, NP_subjects = preprocess(
    "Data_for_Lausanne_30_06_2023_full.xlsx", [], 1,
    patients=0, drop_non_sequenced=False, diag=0, patho="ACMG"
)
X.loc[NP_subjects]

## Univariate baselines

Evaluates two simple clinical rules as comparators:
- **Inhibin-B** alone (lower = more likely CHH)
- **Cryptorchidism − testicular volume** (positive = more likely CHH)

In [ ]:
def process_data(X, y, n_iterations=100, test_size=0.3):
    """
    Evaluate univariate clinical baselines over repeated random splits.

    For each condition/imputation combination, the threshold is set to the
    median of the training score distribution (rank-based, not optimised on
    training labels, to avoid any form of leakage).

    Parameters
    ----------
    X : pd.DataFrame   Feature matrix (NP cohort subjects).
    y : pd.Series      Binary labels.
    n_iterations : int Number of random splits.
    test_size : float  Proportion of data held out as test set.
    """
    imputer     = SimpleImputer(strategy="mean")
    conditions  = ["1st_INB", "Crypt + 1st_TV"]
    imputations = ["mean", "dropna"]
    results = []

    for condition in conditions:
        for imputation in imputations:
            train_auc_scores, test_auc_scores = [], []
            train_bal_acc,    test_bal_acc    = [], []

            for _ in range(n_iterations):
                if imputation == "mean":
                    X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
                else:
                    X_imp = X.dropna()
                y_use = y.reindex(X_imp.index).dropna()

                X_tr, X_te, y_tr, y_te = train_test_split(X_imp, y_use, test_size=test_size, stratify=y_use)

                if condition == "1st_INB":
                    # Lower inhibin-B → more likely CHH → invert sign for AUC
                    s_tr = -X_tr["1st_INB"];  s_te = -X_te["1st_INB"]
                else:
                    # Cryptorchidism (1/0) minus TV: high value → more likely CHH
                    s_tr = X_tr["Crypt"] - X_tr["1st_TV"]
                    s_te = X_te["Crypt"] - X_te["1st_TV"]

                train_auc = roc_auc_score(y_tr, s_tr) if len(np.unique(y_tr)) > 1 else np.nan
                test_auc  = roc_auc_score(y_te, s_te) if len(np.unique(y_te)) > 1 else np.nan

                # Threshold at training-set median (no label-based tuning)
                thresh = np.median(s_tr)
                train_bal_acc.append(balanced_accuracy_score(y_tr, s_tr > thresh))
                test_bal_acc.append( balanced_accuracy_score(y_te, s_te > thresh))
                train_auc_scores.append(train_auc); test_auc_scores.append(test_auc)

            results.append({
                "condition": condition, "imputation": imputation,
                "avg_train_auc":      np.nanmean(train_auc_scores),
                "avg_test_auc":       np.nanmean(test_auc_scores),
                "avg_train_bal_acc":  np.mean(train_bal_acc),
                "avg_test_bal_acc":   np.mean(test_bal_acc),
            })

    for r in results:
        print(f"Condition: {r['condition']}, Imputation: {r['imputation']}")
        print(f"  train AUC: {r['avg_train_auc']:.3f}  |  test AUC: {r['avg_test_auc']:.3f}")
        print(f"  train bal.acc: {r['avg_train_bal_acc']:.3f}  |  test bal.acc: {r['avg_test_bal_acc']:.3f}")
        print()
    return results

In [ ]:
# All patients
X, Y, NP_subjects = preprocess("Data_for_Lausanne_30_06_2023_full.xlsx", [], 1,
                                patients=0, drop_non_sequenced=False, diag=0, patho="ACMG")
X_NP = X.loc[NP_subjects];  y_NP = Y.loc[NP_subjects]
pd.concat([X_NP, y_NP], axis=1).to_excel("preprocessed_Data_Lausanne_DP_ALL_NP.xlsx")
results = process_data(X_NP, y_NP)

In [ ]:
# Partial CHH only (excludes complete CHH subjects)
X, Y, NP_subjects = preprocess("Data_for_Lausanne_30_06_2023_full.xlsx", [], 1,
                                patients=1, drop_non_sequenced=False, diag=0, patho="ACMG")
valid = [s for s in NP_subjects if s in X.index and s in Y.index]
X_NP = X.loc[valid];  y_NP = Y.loc[valid]
pd.concat([X_NP, y_NP], axis=1).to_excel("preprocessed_Data_Lausanne_DP_pCHH.xlsx")
results = process_data(X_NP, y_NP)

In [ ]:
# Complete CHH only (excludes partial CHH subjects)
X, Y, NP_subjects = preprocess("Data_for_Lausanne_30_06_2023_full.xlsx", [], 1,
                                patients=2, drop_non_sequenced=False, diag=0, patho="ACMG")
valid = [s for s in NP_subjects if s in X.index and s in Y.index]
X_NP = X.loc[valid];  y_NP = Y.loc[valid]
pd.concat([X_NP, y_NP], axis=1).to_excel("preprocessed_Data_Lausanne_DP_cCHH.xlsx")
results = process_data(X_NP, y_NP)

## Statistics

Wilcoxon rank-sum tests comparing model variants across all 100 repeated splits.

In [ ]:
global_stats_no_genetics = pd.read_excel("results/global_stats_no_genetics_100_V8_V5_QS_30_NP_only.xlsx")
global_stats_no_rank     = pd.read_excel("results/global_stats_no_rank_100_V8_V5_QS_30_NP_only.xlsx")
global_stats_rf          = pd.read_excel("results/global_stats_rf_100_V8_V5_QS_30_NP_only.xlsx")

### IF (no genetics) vs IF (per-gene score) — does adding genetics help?

In [ ]:
# Overall test across all feature sets
print("Overall ROC-AUC comparison (no_rank > no_genetics):")
print(ranksums(x=global_stats_no_rank["ROC_AUC"],
               y=global_stats_no_genetics["ROC_AUC"],
               alternative="greater"))

# Per-feature-set breakdown
print("\nPer feature set (balanced accuracy):")
for op in global_stats_no_rank["op"].unique():
    stat = ranksums(x=global_stats_no_rank.query(f"op == @op")["ACC"],
                    y=global_stats_no_genetics.query(f"op == @op")["ACC"],
                    alternative="greater")
    print(f"  {op}: {stat}")

### IF (no genetics) vs Random Forest

In [ ]:
print("Overall ACC comparison (no_genetics IF > RF):")
print(ranksums(x=global_stats_no_genetics["ACC"],
               y=global_stats_rf["ACC"],
               alternative="greater"))

print("\nPer feature set:")
for op in global_stats_no_genetics["op"].unique():
    stat = ranksums(x=global_stats_no_genetics.query(f"op == @op")["ACC"],
                    y=global_stats_rf.query(f"op == @op")["ACC"],
                    alternative="greater")
    print(f"  {op}: {stat}")

### Summary tables

In [ ]:
global_stats_no_genetics.groupby('op').agg(['mean','std','sem'])

In [ ]:
global_stats_rf.groupby('op').agg(['mean','std','sem'])

In [ ]:
global_stats_no_rank.groupby('op').agg(['mean','std','sem'])

## SHAP analysis

Fits a single high-fidelity model (1000 trees) on the NP cohort and generates decision plots. Misclassified samples are highlighted.

In [ ]:
X, Y, NP_subjects = preprocess(
    "Data_for_Lausanne_30_06_2023_full.xlsx", [], 1, genetics=1, no_rank=1
)
X_NP = X.loc[intersection(NP_subjects, X.index)].fillna(X.median())
Y_NP = Y.loc[X_NP.index]

X_trainval, X_test, Y_trainval, Y_test = train_test_split(
    X_NP, Y_NP, test_size=0.3, stratify=Y_NP
)

ifc = IsolationForest(n_estimators=1000, n_jobs=-1)
ifc.fit(X_trainval)
predictions = ifc.predict(X_test)

TN, FP, FN, TP = confusion_matrix(
    (((Y_test*2)-1)*-1).astype(int), predictions, labels=[1,-1]
).ravel()
Precision = divide(TP, TP+FP); Recall = divide(TP, TP+FN)
ACC = (divide(TP, TP+FN) + divide(TN, TN+FP)) / 2
print(f"ACC: {ACC:.2f}  Precision: {Precision:.2f}  Recall: {Recall:.2f}")

explainer   = shap.TreeExplainer(ifc)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Decision plot — highlight misclassified subjects
# Prediction sign: negative SHAP sum → anomaly (CHH); positive → inlier (CDGP)
y_pred        = shap_values.sum(1) < 0
misclassified = y_pred != Y_test.values

shap.decision_plot(explainer.expected_value, shap_values, X_test.columns,
                   highlight=misclassified, show=False)
f = plt.gcf()
f.savefig("./figures/SHAP_test_clinical_genetics_V8.svg")
f.savefig("./figures/SHAP_test_clinical_genetics_V8.png")

In [ ]:
# Force plots for individual subjects (first 3 shown; extend index as needed)
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], X_test.columns)

In [ ]:
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[1], X_test.columns)

In [ ]:
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[2], X_test.columns)